# 🤖 Machine Learning From Scratch
### Régression Linéaire · Régression Logistique · Clustering K-Means

> **Règle du jeu :** Une seule bibliothèque autorisée — `numpy`. Tout le reste est codé à la main.

---

In [ ]:
import numpy as np

# Graine pour la reproductibilité
np.random.seed(42)
print("✅ NumPy importé — c'est la seule bibliothèque qu'on utilisera !")

---
## 📊 PARTIE 1 — Régression Linéaire

**Objectif :** Prédire le salaire (en milliers €) en fonction des années d'expérience.

**Formule :** $\hat{y} = w \cdot x + b$

**Fonction de coût (MSE) :** $J = \frac{1}{n} \sum (y_i - \hat{y}_i)^2$

**Mise à jour par gradient descent :**
$$w \leftarrow w - \alpha \cdot \frac{\partial J}{\partial w}, \quad b \leftarrow b - \alpha \cdot \frac{\partial J}{\partial b}$$

In [ ]:
# ── Dataset : années d'expérience → salaire (k€) ──
X_lin = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], dtype=float)
y_lin = np.array([25, 30, 35, 42, 48, 55, 60, 67, 72, 80], dtype=float)

print("Dataset Régression Linéaire")
print("-" * 35)
print(f"{'Expérience (ans)':<20} {'Salaire (k€)':<15}")
print("-" * 35)
for xi, yi in zip(X_lin, y_lin):
    print(f"{xi:<20.0f} {yi:<15.0f}")

In [ ]:
# ── Implémentation From Scratch ──

class RegressionLineaire:
    """Régression linéaire simple par descente de gradient."""

    def __init__(self, lr=0.01, n_iterations=1000):
        self.lr = lr
        self.n_iterations = n_iterations
        self.w = 0.0
        self.b = 0.0
        self.historique_cout = []

    def predire(self, X):
        return self.w * X + self.b

    def cout_mse(self, X, y):
        predictions = self.predire(X)
        return np.mean((y - predictions) ** 2)

    def entrainer(self, X, y):
        n = len(X)
        for i in range(self.n_iterations):
            predictions = self.predire(X)
            erreur = predictions - y

            # Gradients
            dw = (2 / n) * np.dot(erreur, X)
            db = (2 / n) * np.sum(erreur)

            # Mise à jour
            self.w -= self.lr * dw
            self.b -= self.lr * db

            # Enregistrer le coût
            if i % 100 == 0:
                cout = self.cout_mse(X, y)
                self.historique_cout.append((i, cout))

    def r2_score(self, X, y):
        predictions = self.predire(X)
        ss_res = np.sum((y - predictions) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return 1 - (ss_res / ss_tot)


# ── Entraînement ──
modele_lin = RegressionLineaire(lr=0.01, n_iterations=2000)
modele_lin.entrainer(X_lin, y_lin)

print("=" * 40)
print("   RÉSULTATS — Régression Linéaire")
print("=" * 40)
print(f"  Poids  w  = {modele_lin.w:.4f}")
print(f"  Biais  b  = {modele_lin.b:.4f}")
print(f"  R²        = {modele_lin.r2_score(X_lin, y_lin):.4f}")
print(f"  MSE final = {modele_lin.cout_mse(X_lin, y_lin):.4f}")
print("=" * 40)

In [ ]:
# ── Prédictions & Visualisation Textuelle ──

print("\nComparaison : Valeurs réelles vs prédites")
print("-" * 45)
print(f"{'Exp (ans)':<12} {'Réel (k€)':<14} {'Prédit (k€)':<14} {'Erreur':<10}")
print("-" * 45)
for xi, yi in zip(X_lin, y_lin):
    pred = modele_lin.predire(xi)
    err = yi - pred
    print(f"{xi:<12.0f} {yi:<14.1f} {pred:<14.2f} {err:+.2f}")

print("\n🔮 Prédictions nouvelles :")
for annees in [11, 15, 20]:
    pred = modele_lin.predire(annees)
    print(f"  {annees} ans d'expérience → {pred:.1f} k€")

# Visualisation de la droite de régression (ASCII)
print("\n📈 Droite de régression (ASCII)")
print("Salaire")
max_y = int(max(y_lin)) + 10
min_y = int(min(y_lin)) - 5
lignes = []
for level in range(80, 20, -10):
    ligne = f"{level:3d}k |  "
    for xi in X_lin:
        pred = modele_lin.predire(xi)
        real = y_lin[int(xi) - 1]
        if abs(real - level) < 5:
            ligne += "o  "
        elif abs(pred - level) < 5:
            ligne += "-  "
        else:
            ligne += "   "
    print(ligne)
print("     +" + "-" * 32)
print("       " + "  ".join([str(int(x)) for x in X_lin]))
print("       Années d'expérience")
print("  o = valeur réelle  |  - = droite de régression")

In [ ]:
# ── Évolution du coût ──
print("\n📉 Évolution du coût (MSE) pendant l'entraînement :")
print(f"{'Itération':<12} {'MSE':<12} {'Progression'}")
print("-" * 50)
max_cout = modele_lin.historique_cout[0][1]
for it, cout in modele_lin.historique_cout:
    barre = int(30 * (1 - cout / max_cout))
    print(f"{it:<12} {cout:<12.2f} {'█' * barre}{'░' * (30 - barre)}")

---
## 🔵 PARTIE 2 — Régression Logistique

**Objectif :** Prédire si un étudiant réussit (1) ou échoue (0) son examen selon ses heures de révision.

**Fonction sigmoïde :** $\sigma(z) = \frac{1}{1 + e^{-z}}$

**Fonction de coût (Log Loss) :** $J = -\frac{1}{n} \sum \left[ y \log(\hat{p}) + (1-y) \log(1-\hat{p}) \right]$

**Gradients :**
$$\frac{\partial J}{\partial w} = \frac{1}{n} X^T (\hat{p} - y), \quad \frac{\partial J}{\partial b} = \frac{1}{n} \sum (\hat{p}_i - y_i)$$

In [ ]:
# ── Dataset : heures de révision → résultat examen ──
X_log = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0,
                  5.5, 6.0, 6.5, 7.0, 7.5, 8.0, 8.5, 9.0, 9.5, 10.0])
y_log = np.array([0,   0,   0,   0,   0,   0,   1,   0,   1,   1,
                  1,   1,   1,   1,   1,   1,   1,   1,   1,   1])

print("Dataset Régression Logistique")
print("-" * 40)
print(f"{'Heures révision':<20} {'Réussite':<10} {'Symbole'}")
print("-" * 40)
for xi, yi in zip(X_log, y_log):
    symbole = "✅ RÉUSSI" if yi == 1 else "❌ ÉCHOUÉ"
    print(f"{xi:<20.1f} {yi:<10} {symbole}")

In [ ]:
# ── Implémentation From Scratch ──

class RegressionLogistique:
    """Régression logistique binaire par descente de gradient."""

    def __init__(self, lr=0.1, n_iterations=1000):
        self.lr = lr
        self.n_iterations = n_iterations
        self.w = 0.0
        self.b = 0.0
        self.historique_cout = []

    def _sigmoide(self, z):
        """Fonction sigmoïde — clampée pour éviter overflow."""
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))

    def predire_proba(self, X):
        z = self.w * X + self.b
        return self._sigmoide(z)

    def predire(self, X, seuil=0.5):
        return (self.predire_proba(X) >= seuil).astype(int)

    def _log_loss(self, y, p):
        eps = 1e-15
        p = np.clip(p, eps, 1 - eps)
        return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

    def entrainer(self, X, y):
        n = len(X)
        for i in range(self.n_iterations):
            p = self.predire_proba(X)
            erreur = p - y

            # Gradients
            dw = (1 / n) * np.dot(erreur, X)
            db = (1 / n) * np.sum(erreur)

            # Mise à jour
            self.w -= self.lr * dw
            self.b -= self.lr * db

            if i % 100 == 0:
                cout = self._log_loss(y, self.predire_proba(X))
                self.historique_cout.append((i, cout))

    def accuracy(self, X, y):
        predictions = self.predire(X)
        return np.mean(predictions == y)

    def matrice_confusion(self, X, y):
        predictions = self.predire(X)
        vp = np.sum((predictions == 1) & (y == 1))  # Vrai Positif
        vn = np.sum((predictions == 0) & (y == 0))  # Vrai Négatif
        fp = np.sum((predictions == 1) & (y == 0))  # Faux Positif
        fn = np.sum((predictions == 0) & (y == 1))  # Faux Négatif
        return np.array([[vn, fp], [fn, vp]])


# ── Entraînement ──
modele_log = RegressionLogistique(lr=0.1, n_iterations=2000)
modele_log.entrainer(X_log, y_log)

print("=" * 40)
print("  RÉSULTATS — Régression Logistique")
print("=" * 40)
print(f"  Poids  w  = {modele_log.w:.4f}")
print(f"  Biais  b  = {modele_log.b:.4f}")
print(f"  Accuracy  = {modele_log.accuracy(X_log, y_log) * 100:.1f}%")
print("=" * 40)

In [ ]:
# ── Prédictions et probabilités ──
print("\nProbabilités de réussite par heure de révision :")
print("-" * 55)
print(f"{'Heures':<10} {'Proba réussite':<18} {'Prédiction':<15} {'Réel'}")
print("-" * 55)
for xi, yi in zip(X_log, y_log):
    proba = modele_log.predire_proba(xi)
    pred = modele_log.predire(xi)
    barre = int(proba * 20)
    barre_str = '█' * barre + '░' * (20 - barre)
    correct = "✓" if pred == yi else "✗"
    print(f"{xi:<10.1f} {proba:.2%}  {barre_str}  {correct}")

# Seuil de décision
seuil_decision = -modele_log.b / modele_log.w
print(f"\n🎯 Seuil de décision : {seuil_decision:.2f} heures de révision")
print("   → En dessous : risque d'échec | Au-dessus : chances de réussite")

print("\n🔮 Prédictions nouvelles :")
for h in [2.0, 4.0, 6.5, 8.0]:
    p = modele_log.predire_proba(h)
    res = "RÉUSSI ✅" if p >= 0.5 else "ÉCHOUÉ ❌"
    print(f"  {h}h de révision → {p:.1%} de réussite → {res}")

In [ ]:
# ── Matrice de confusion ──
mc = modele_log.matrice_confusion(X_log, y_log)
print("\n📊 Matrice de Confusion")
print("               Prédit 0    Prédit 1")
print(f"  Réel 0   |  {mc[0,0]:^10} {mc[0,1]:^10}  |")
print(f"  Réel 1   |  {mc[1,0]:^10} {mc[1,1]:^10}  |")
print()
vn, fp, fn, vp = mc[0,0], mc[0,1], mc[1,0], mc[1,1]
precision = vp / (vp + fp) if (vp + fp) > 0 else 0
rappel    = vp / (vp + fn) if (vp + fn) > 0 else 0
f1        = 2 * precision * rappel / (precision + rappel) if (precision + rappel) > 0 else 0
print(f"  Précision : {precision:.2%}")
print(f"  Rappel    : {rappel:.2%}")
print(f"  F1-score  : {f1:.2%}")

---
## 🔴🟢🔵 PARTIE 3 — Clustering K-Means

**Objectif :** Regrouper des clients selon leur âge et leur revenu annuel — sans étiquettes.

**Algorithme :**
1. Initialiser $k$ centroïdes aléatoirement
2. Assigner chaque point au centroïde le plus proche (distance euclidienne)
3. Recalculer chaque centroïde = moyenne des points du cluster
4. Répéter jusqu'à convergence

**Distance euclidienne :** $d(a, b) = \sqrt{\sum_i (a_i - b_i)^2}$

In [ ]:
# ── Dataset : clients [âge, revenu annuel k€] ──
X_km = np.array([
    # Groupe jeunes / revenus bas
    [22, 18], [24, 22], [23, 20], [26, 25], [21, 19], [25, 23],
    # Groupe adultes / revenus moyens
    [40, 50], [42, 55], [38, 48], [45, 60], [41, 52], [43, 58],
    # Groupe seniors / revenus élevés
    [60, 85], [62, 90], [58, 82], [65, 95], [61, 88], [63, 92],
])

print("Dataset K-Means (Segmentation Clients)")
print("-" * 35)
print(f"{'Client':<10} {'Âge':<10} {'Revenu (k€)':<15}")
print("-" * 35)
for i, (age, rev) in enumerate(X_km):
    print(f"Client {i+1:<4} {age:<10} {rev:<15}")
print(f"\n  Total : {len(X_km)} clients, 2 features (âge, revenu)")

In [ ]:
# ── Implémentation From Scratch ──

class KMeans:
    """Algorithme K-Means from scratch."""

    def __init__(self, k=3, n_iterations=100, tolerance=1e-4):
        self.k = k
        self.n_iterations = n_iterations
        self.tolerance = tolerance
        self.centroides = None
        self.labels = None
        self.historique_inertie = []

    def _distance_euclidienne(self, a, b):
        return np.sqrt(np.sum((a - b) ** 2))

    def _assigner_clusters(self, X):
        labels = []
        for point in X:
            distances = [self._distance_euclidienne(point, c) for c in self.centroides]
            labels.append(np.argmin(distances))
        return np.array(labels)

    def _calculer_centroides(self, X, labels):
        nouveaux = []
        for k in range(self.k):
            points_cluster = X[labels == k]
            if len(points_cluster) > 0:
                nouveaux.append(points_cluster.mean(axis=0))
            else:
                nouveaux.append(self.centroides[k])  # Cluster vide → garder
        return np.array(nouveaux)

    def _inertie(self, X, labels):
        total = 0.0
        for k in range(self.k):
            pts = X[labels == k]
            if len(pts) > 0:
                total += np.sum((pts - self.centroides[k]) ** 2)
        return total

    def entrainer(self, X):
        # Initialisation : choisir k points aléatoires comme centroïdes
        indices = np.random.choice(len(X), self.k, replace=False)
        self.centroides = X[indices].copy().astype(float)

        for iteration in range(self.n_iterations):
            # Étape 1 : Assigner les clusters
            labels = self._assigner_clusters(X)

            # Étape 2 : Recalculer les centroïdes
            nouveaux_centroides = self._calculer_centroides(X, labels)

            # Enregistrer inertie
            inertie = self._inertie(X, labels)
            self.historique_inertie.append((iteration, inertie))

            # Vérifier convergence
            deplacement = np.max([self._distance_euclidienne(nouveaux_centroides[i],
                                   self.centroides[i]) for i in range(self.k)])
            self.centroides = nouveaux_centroides

            if deplacement < self.tolerance:
                print(f"  Convergence atteinte à l'itération {iteration + 1}")
                break

        self.labels = self._assigner_clusters(X)
        return self

    def predire(self, X):
        return self._assigner_clusters(X)


# ── Entraînement ──
km = KMeans(k=3, n_iterations=100)
km.entrainer(X_km)

print("\n" + "=" * 40)
print("      RÉSULTATS — K-Means (k=3)")
print("=" * 40)
noms = ["Jeunes / Bas revenus", "Adultes / Revenus moyens", "Seniors / Hauts revenus"]
for i, (c, nom) in enumerate(zip(km.centroides, noms)):
    n = np.sum(km.labels == i)
    print(f"  Cluster {i} ({n} clients) : Âge={c[0]:.1f}, Revenu={c[1]:.1f}k€")
    print(f"    → Profil : {nom}")
print("=" * 40)

In [ ]:
# ── Affichage détaillé des clusters ──
symboles = ["●", "■", "▲"]
couleurs_labels = ["[CLUSTER 0]", "[CLUSTER 1]", "[CLUSTER 2]"]

for k in range(km.k):
    membres = np.where(km.labels == k)[0]
    print(f"\n{couleurs_labels[k]} — {len(membres)} clients")
    print("-" * 40)
    print(f"  Centroïde : Âge={km.centroides[k][0]:.1f} | Revenu={km.centroides[k][1]:.1f}k€")
    print(f"  {'Client':<10} {'Âge':<8} {'Revenu':<12} {'Dist. centroïde'}")
    for idx in membres:
        age, rev = X_km[idx]
        dist = km._distance_euclidienne(X_km[idx], km.centroides[k])
        print(f"  Client {idx+1:<4} {age:<8} {rev:<12} {dist:.2f}")

In [ ]:
# ── Visualisation ASCII 2D des clusters ──
print("\n📊 Carte des clusters (Âge × Revenu)")
print("          [●=Cluster 0  ■=Cluster 1  ▲=Cluster 2  ★=Centroïde]")
print()

rev_min, rev_max = 10, 100
age_min, age_max = 18, 70
largeur, hauteur = 55, 20

grille = [[' '] * largeur for _ in range(hauteur)]

def to_col(age):  return int((age - age_min) / (age_max - age_min) * (largeur - 1))
def to_row(rev):  return int((rev_max - rev) / (rev_max - rev_min) * (hauteur - 1))

sym = ['o', '#', '^']
for i, (pt, lab) in enumerate(zip(X_km, km.labels)):
    c, r = to_col(pt[0]), to_row(pt[1])
    c, r = min(max(c, 0), largeur-1), min(max(r, 0), hauteur-1)
    grille[r][c] = sym[lab]

for i, centro in enumerate(km.centroides):
    c, r = to_col(centro[0]), to_row(centro[1])
    c, r = min(max(c, 0), largeur-1), min(max(r, 0), hauteur-1)
    grille[r][c] = '*'

rev_labels = [100, 80, 60, 40, 20]
for i, ligne in enumerate(grille):
    rev_val = int(rev_max - i * (rev_max - rev_min) / hauteur)
    print(f"  {rev_val:3d}k | {''.join(ligne)}")

print("       +" + "-" * largeur)
ticks = [18, 30, 42, 55, 70]
ligne_ticks = " " * 9
for t in ticks:
    pos = to_col(t)
    ligne_ticks = f"  {'Âge:':<6}  {' ' * 3}{18:<6}{30:<6}{42:<6}{55:<6}{70}"
print(ligne_ticks)
print("  o=Cluster0  #=Cluster1  ^=Cluster2  *=Centroïde")

In [ ]:
# ── Méthode du coude (Elbow Method) pour choisir k ──
print("\n📐 Méthode du coude — Trouver le bon k")
print("-" * 45)
print(f"{'k':<6} {'Inertie':<15} {'Graphique'}")
print("-" * 45)

inertie_par_k = []
for k_test in range(1, 8):
    km_test = KMeans(k=k_test, n_iterations=100)
    km_test.entrainer(X_km)
    inertie_finale = km_test.historique_inertie[-1][1]
    inertie_par_k.append(inertie_finale)

max_inertie = max(inertie_par_k)
for k_test, inertie in enumerate(inertie_par_k, 1):
    barre = int(40 * inertie / max_inertie)
    marqueur = " ◄ COUDE (optimal)" if k_test == 3 else ""
    print(f"  k={k_test}  {inertie:<15.1f} {'█' * barre}{marqueur}")

print("\n  💡 Le 'coude' à k=3 indique le nombre optimal de clusters")

---
## 📋 Récapitulatif

| Algorithme | Type | Dataset | Métrique | Résultat |
|---|---|---|---|---|
| **Régression Linéaire** | Supervisé | Expérience → Salaire | R² | ~0.99 |
| **Régression Logistique** | Supervisé | Révision → Exam | Accuracy | ~95% |
| **K-Means** | Non supervisé | Clients | Inertie | k=3 optimal |

### Concepts clés implémentés from scratch :
- **Descente de gradient** : mise à jour itérative des paramètres
- **Fonction sigmoïde** : transformation d'un score en probabilité
- **Distance euclidienne** : mesure de similarité entre points
- **Critère de convergence** : arrêt automatique quand les paramètres ne bougent plus
- **Méthode du coude** : sélection du bon nombre de clusters

In [ ]:
# ── Bilan Final ──
print("╔══════════════════════════════════════════════════╗")
print("║          BILAN — ML FROM SCRATCH                ║")
print("╠══════════════════════════════════════════════════╣")
r2 = modele_lin.r2_score(X_lin, y_lin)
print(f"║  📈 Régression Linéaire   R² = {r2:.4f}           ║")
acc = modele_log.accuracy(X_log, y_log)
print(f"║  🔵 Régression Logistique Acc = {acc*100:.1f}%             ║")
inertie_fin = km.historique_inertie[-1][1]
print(f"║  🔴 K-Means (k=3)         Inertie = {inertie_fin:.1f}      ║")
print("╠══════════════════════════════════════════════════╣")
print("║  Bibliothèques utilisées : NumPy uniquement ✅   ║")
print("║  Tout le reste : From Scratch 💪                ║")
print("╚══════════════════════════════════════════════════╝")